# Tarea 2 Parte A
Integrantes:
- Carlos Raúl Sánchez Figueroa
- Ulises Omar Montes Correa
- Diego Córdoba Gómez
- Guillermo Collado

#Inciso 1)

In [0]:
import pyspark.sql.functions as F

In [0]:
df_silver = spark.table("dev.ciencias_data.silver_sessions")
display(df_silver)

##Tamaño y estructura de los datos

In [0]:
n_rows = df_silver.count()
n_cols = len(df_silver.columns)

print(f"Número de registros: {n_rows}")
print(f"Número de columnas: {n_cols}")

df_silver.printSchema()

###Interpretación

La tabla silver contiene aproximadamente 29,552 registros con múltiples variables que describen el tráfico de red.
Los tipos de datos son consistentes con la naturaleza de cada variable, permitiendo su análisis posterior.

##Duplicados

In [0]:
total = df_silver.count()
sin_dup = df_silver.dropDuplicates().count()

duplicados = total - sin_dup

print(f"Duplicados: {duplicados}")

###Interpretación

No se identificaron registros duplicados en el dataset, lo cual garantiza la integridad de los datos y evita sesgos en el análisis.

##Análisis de valores nulos

In [0]:
df_nulls = df_silver.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_silver.columns
])

df_nulls_long = df_nulls.select(
    F.explode(
        F.array([
            F.struct(F.lit(c).alias("columna"), F.col(c).alias("nulos"))
            for c in df_nulls.columns
        ])
    ).alias("tmp")
).select("tmp.*")

display(df_nulls_long)

Databricks visualization. Run in Databricks to view.

###Interpretación

Se identificó una alta proporción de valores nulos en variables como src_asn, dst_asn, src_geo, dst_geo e init_rtt.
Esto indica que la información de geolocalización, sistema autónomo y latencia no está disponible para una gran parte de las sesiones.
Por lo tanto, estas variables no se consideran críticas para el análisis posterior.

##Análisis de valores cero

In [0]:
df_zeros = df_silver.select(
    F.count(F.when(F.col("tot_bytes") == 0, True)).alias("tot_bytes"),
    F.count(F.when(F.col("tot_packets") == 0, True)).alias("tot_packets"),
    F.count(F.when(F.col("tot_data_bytes") == 0, True)).alias("tot_data_bytes")
)

df_zeros_long = df_zeros.select(
    F.explode(
        F.array([
            F.struct(F.lit("tot_bytes").alias("variable"), F.col("tot_bytes").alias("zeros")),
            F.struct(F.lit("tot_packets").alias("variable"), F.col("tot_packets").alias("zeros")),
            F.struct(F.lit("tot_data_bytes").alias("variable"), F.col("tot_data_bytes").alias("zeros"))
        ])
    ).alias("tmp")
).select("tmp.*")

display(df_zeros_long)

Databricks visualization. Run in Databricks to view.

###Interpretación

Se observaron 5430 registros con valor cero en tot_data_bytes, lo cual puede representar sesiones sin transferencia efectiva de datos.
Este comportamiento es consistente con tráfico de red y no se considera un error.

##Validación de consistencia

In [0]:
df_consistency = df_silver.withColumn(
    "diff_bytes",
    F.col("tot_bytes") - (F.col("src_bytes") + F.col("dst_bytes"))
)

display(df_consistency.select("diff_bytes"))

Databricks visualization. Run in Databricks to view.

###Interpretación

La diferencia entre tot_bytes y la suma de src_bytes y dst_bytes es igual a cero en todos los registros, lo cual confirma que los datos son completamente consistentes.

##Estadísticas descriptivas

In [0]:
display(df_silver.select("tot_bytes"))
display(df_silver.select("tot_packets"))
display(df_silver.select("tot_data_bytes"))

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

###Interpretación

Se observa una alta dispersión en las variables numéricas, especialmente en tot_bytes, donde existen valores extremos significativamente mayores al promedio.
Esto indica la presencia de sesiones de alto volumen, lo cual es esperado en tráfico de red.

##Conclusión general
Se realizó un análisis de calidad de los datos en la capa silver, evaluando duplicados, valores nulos, valores cero, consistencia y estadísticas descriptivas.
No se encontraron registros duplicados, lo cual garantiza la integridad del dataset.
Se identificó una alta proporción de valores nulos en variables relacionadas con geolocalización y sistema autónomo, por lo que su uso en el análisis será limitado.
Las variables principales de tráfico (tot_bytes, tot_packets, tot_data_bytes) presentan buena calidad y consistencia.
Se detectaron valores cero en tot_data_bytes, los cuales son coherentes con la naturaleza del tráfico de red.
Asimismo, se observó una alta dispersión en las variables numéricas, con presencia de valores extremos esperados en este tipo de datos.
Finalmente, se confirmó la consistencia total de los datos, lo que permite continuar con confianza hacia la etapa de Feature Engineering y modelado.

#Inciso 2)

In [0]:
import pyspark.sql.functions as F

df_silver = spark.table("dev.ciencias_data.silver_sessions")

In [0]:
df_silver = df_silver.withColumn(
    "session_duration",
    F.col("last_packet").cast("long") - F.col("first_packet").cast("long")
)

In [0]:
df_features = df_silver.withColumn(
    "bytes_per_second",
    F.when(F.col("session_duration") > 0,
           F.col("tot_bytes") / F.col("session_duration"))
).withColumn(
    "avg_packet_size",
    F.when(F.col("tot_packets") > 0,
           F.col("tot_bytes") / F.col("tot_packets"))
).withColumn(
    "bytes_ratio_src_dst",
    F.when(F.col("dst_bytes") > 0,
           F.col("src_bytes") / F.col("dst_bytes"))
).withColumn(
    "packets_ratio_src_dst",
    F.when(F.col("dst_packets") > 0,
           F.col("src_packets") / F.col("dst_packets"))
)

In [0]:
df_features = df_features.select(
    "community_id",
    "src_ip",
    "dst_ip",
    "protocol",
    "tot_bytes",
    "tot_packets",
    "tot_data_bytes",
    "session_duration",
    "bytes_per_second",
    "avg_packet_size",
    "bytes_ratio_src_dst",
    "packets_ratio_src_dst"
)

In [0]:
df_features.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dev.ciencias_data.fs_sessions")

In [0]:
display(spark.table("dev.ciencias_data.fs_sessions"))

###Feature Engineering

Se construyeron nuevas variables a partir de los datos originales con el objetivo de capturar mejor el comportamiento del tráfico de red.
Entre las principales variables generadas se encuentran:

bytes_per_second: mide la velocidad de transmisión de datos por sesión
avg_packet_size: representa el tamaño promedio de los paquetes
bytes_ratio_src_dst: indica la relación entre bytes enviados y recibidos
packets_ratio_src_dst: mide la proporción de paquetes entre origen y destino

Estas variables permiten caracterizar de manera más precisa cada sesión, facilitando la identificación de patrones y anomalías en el tráfico de red.

Finalmente, las features fueron almacenadas en una tabla Delta dentro de Unity Catalog, funcionando como un repositorio centralizado de variables reutilizables para modelos de machine learning.